# M4: pilot QLoRA trên Colab
Chạy sau khi SFT đã được duyệt. Chọn GPU. Artifact trong Drive, không đưa vào Git.
Dependency chưa kiểm chứng trên GPU của bạn; cell cài đặt lưu pip freeze.
Mặc định 100 bước pilot, không train hết pool 10000 trong thử nghiệm này.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import sys

REPO_DIR = Path('/content/kids-story')
SFT_DIR = Path('/content/drive/MyDrive/kids-story-runs/m2/REPLACE_WITH_APPROVED_SFT')
RUN_DIR = Path('/content/drive/MyDrive/kids-story-runs/m4') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/coder-fishing/KidStory.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
config = {
    'sft_dir': str(SFT_DIR),
    'output_dir': str(RUN_DIR),
    'seed': 42,
    'max_steps': 100,
    'max_length': 2048,
    'resume_from_checkpoint': None,
}
CONFIG_PATH = Path('/content/m4_config.json')
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding='utf-8')


## 1. Kiểm tra SFT trên CPU trước khi cài GPU dependencies

In [ ]:
subprocess.run([
    sys.executable, '-m', 'kids_story.training',
    str(CONFIG_PATH), '--preflight-only',
], check=True)


## 2. Cài môi trường GPU
Chỉ chạy cell này một lần mỗi runtime. Nếu pip yêu cầu restart, restart rồi chạy lại cell cấu hình và bỏ qua cell cài này.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'unsloth', 'datasets', 'peft', 'accelerate', 'bitsandbytes',
], check=True)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
freeze_path = RUN_DIR.parent / (RUN_DIR.name + '_environment.txt')
freeze_path.parent.mkdir(parents=True, exist_ok=True)
freeze_path.write_text(freeze, encoding='utf-8')


## 3. Train pilot
Xem token_label_audit.json: prompt có labels -100; response có nhãn. Quá giới hạn 2048 token sẽ dừng để bạn điều chỉnh cấu hình, không âm thầm cắt truyện. Resume: đặt đường dẫn checkpoint cũ ở cấu hình và dùng RUN_DIR mới.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'kids_story.training', str(CONFIG_PATH),
], check=True)
print('Kết quả:', RUN_DIR)


## 4. Nạp adapter trong process mới và sinh dev
Không mở golden/final test. Chấm chất lượng sau khi có outputs, cùng protocol với baseline.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'kids_story.pilot_eval',
    '--run-dir', str(RUN_DIR),
], check=True)
print((RUN_DIR / 'dev_reload/COMPLETE.json').read_text(encoding='utf-8'))
